In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV,KFold, StratifiedKFold,cross_val_score
from lightgbm import LGBMClassifier
import xgboost as xgb
from catboost import CatBoostClassifier

In [ ]:
df_train=pd.read_csv('/kaggle/input/competitions/playground-series-s6e8/train.csv')
df_test=pd.read_csv('/kaggle/input/competitions/playground-series-s6e8/test.csv')

In [ ]:
x_train=df_train.drop(columns=['id','addicted_label'])
y_train=df_train['addicted_label']
x_test=df_test.drop(columns=['id'])

In [ ]:
def prepare_data(X_train, y_train, n_iterations=3):
    """
    Performs:
    1. Basic numerical EDA
    2. Feature + target correlation matrix
    3. Initial median imputation for numerical columns
    4. Iterative XGBoost imputation for numerical columns
    5. Returns the modified X_train

    """

    X = X_train.copy()
    y = y_train.copy()

    print("=" * 60)
    print("DATASET SHAPE")
    print("=" * 60)
    print(X.shape)

    print("\nMissing values:")
    print(
        X.isnull()
         .sum()
         .sort_values(ascending=False)
         .loc[lambda x: x > 0]
    )

    numerical_cols = X.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_cols = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()
    for col in categorical_cols:
        proportions = pd.crosstab(
            X[col],
            y,
            normalize="index"
        )
        proportions.plot(
            kind="bar",
            stacked=True,
            figsize=(8, 5)
        )
        plt.xlabel(col)
        plt.ylabel("Proportion")
        plt.title(f"{col} vs Target Distribution")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

    # Convert object columns to categorical
    for col in categorical_cols:
        X[col] = X[col].astype("category")

    if numerical_cols:
        X[numerical_cols].hist(
            figsize=(15, 12),
            bins=30
        )

        plt.tight_layout()
        plt.show()

    eda_df = X[numerical_cols].copy()
    eda_df["TARGET"] = y.values

    corr = eda_df.corr()

    plt.figure(figsize=(14, 10))

    sns.heatmap(
        corr,
        cmap="coolwarm",
        center=0,
        annot=False
    )

    plt.title("Feature + Target Correlation Matrix")
    plt.tight_layout()
    plt.show()
    impute_cols = [
        col for col in numerical_cols
        if X[col].isna().any()
    ]

    for col in numerical_cols:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(
        x=X[col],
        y=y,
        alpha=0.4
        )
        plt.xlabel(col)
        plt.ylabel("Target")
        plt.title(f"{col} vs Target")
        plt.tight_layout()
        plt.show()

    if not impute_cols:
        print("\nNo numerical missing values found.")
        return X

    print("\nColumns requiring imputation:")
    print(impute_cols)
    print("\nRemaining numerical missing values:")

    print(
        X[numerical_cols]
        .isnull()
        .sum()
        .loc[lambda x: x > 0]
    )

    return X

In [ ]:
x_train=prepare_data(x_train,y_train)

In [ ]:
def prepare_test_data(X_test,n_iterations=3):
    """
    Performs:
    1. Basic numerical EDA
    2. Feature + target correlation matrix
    3. Initial median imputation for numerical columns
    4. Iterative XGBoost imputation for numerical columns
    5. Returns the modified X_train

    """
    X = X_test.copy()
    numerical_cols = X.select_dtypes(
        include=np.number
    ).columns.tolist()
    impute_cols = [
        col for col in numerical_cols
        if X[col].isna().any()
    ]
    categorical_cols = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()
    for col in categorical_cols:
        X[col] = X[col].astype("category")
    if not impute_cols:
        print("\nNo numerical missing values found.")
        return X

    print("\nColumns requiring imputation:")
    print(impute_cols)
    print("\nRemaining numerical missing values:")
    return X

In [ ]:
x_test=prepare_test_data(x_test)

In [ ]:
def evaluate_features(
    X,
    y,
    task,
    cv=3,
    random_state=42
):
    """
    Quickly evaluate a feature set using fixed-model CV.

    task:
        "regression" or "classification"

    Returns:
        mean_score, std_score
    """

    if task == "regression":
            model = xgb.XGBRegressor(
                tree_method='hist',
                n_estimators=300,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                random_state=random_state,
                n_jobs=-1,
                enable_categorical=True
            )

            cv_splitter = KFold(
            n_splits=cv,
            shuffle=True,
            random_state=random_state
            )

            scoring = "neg_root_mean_squared_error"

    elif task == "classification":
            model = xgb.XGBClassifier(
                tree_method='hist',
                n_estimators=300,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                random_state=random_state,
                n_jobs=-1,
                enable_categorical=True
            )

            cv_splitter = StratifiedKFold(
            n_splits=cv,
            shuffle=True,
            random_state=random_state
            )

            scoring = "roc_auc"

    else:
        raise ValueError(
            "task must be 'regression' or 'classification'"
        )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv_splitter,
        scoring=scoring,
        n_jobs=-1
    )
    model.fit(X, y)
    importance = pd.Series(
        model.feature_importances_,
        index=X.columns
    )
    importance_percentage = (
        importance / importance.sum() * 100
    ).sort_values(ascending=False)
    plt.figure(figsize=(10, max(6, len(importance_percentage) * 0.3)))
    sns.barplot(
        x=importance_percentage.values,
        y=importance_percentage.index
    )
    plt.xlabel("Feature Importance (%)")
    plt.ylabel("Feature")
    plt.title("XGBoost Feature Importance")
    plt.tight_layout()
    plt.show()
    if task == "regression":
        scores = -scores
    return scores.mean(), scores.std()

**Baseline Performance**

In [ ]:
evaluate_features(x_train,y_train,task='classification')

In [ ]:
x_temp=x_train.copy()
x_temp['total_screen_week']=(x_temp['daily_screen_time_hours']*5 + x_temp['weekend_screen_time']*2)
x_temp['sleep_share']=x_temp['sleep_hours']/(24-x_temp['daily_screen_time_hours']-x_temp['work_study_hours']).clip(lower=0.001)
x_temp['dopamine_ratio']=(x_temp['gaming_hours']+x_temp['social_media_hours'])/(x_temp['daily_screen_time_hours']+1e-5)
x_temp['notif_vs_app']=x_temp['notifications_per_day']/(x_temp['app_opens_per_day']+1e-5)

In [ ]:
evaluate_features(x_temp,y_train,task='classification')

In [ ]:
x_train=x_temp

In [ ]:
x_test['total_screen_week']=(x_test['daily_screen_time_hours']*5 + x_test['weekend_screen_time']*2)
x_test['sleep_share']=x_test['sleep_hours']/(24-x_test['daily_screen_time_hours']-x_test['work_study_hours']).clip(lower=0.001)
x_test['dopamine_ratio']=(x_test['gaming_hours']+x_test['social_media_hours'])/(x_test['daily_screen_time_hours']+1e-5)
x_test['notif_vs_app']=x_test['notifications_per_day']/(x_test['app_opens_per_day']+1e-5)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder

In [ ]:
trf1=make_column_transformer((OneHotEncoder(handle_unknown='ignore',sparse_output=False),['gender','academic_work_impact']),(OrdinalEncoder(categories=[['Low','Medium','High']],handle_unknown='use_encoded_value',unknown_value=np.nan,encoded_missing_value=np.nan),['stress_level']),remainder='passthrough',verbose_feature_names_out=False)

In [ ]:
trf1.set_output(transform="pandas")
x_train=trf1.fit_transform(x_train)

In [ ]:
trf1_test=make_column_transformer((OneHotEncoder(handle_unknown='ignore',sparse_output=False),['gender','academic_work_impact']),(OrdinalEncoder(categories=[['Low','Medium','High']],handle_unknown='use_encoded_value',unknown_value=np.nan,encoded_missing_value=np.nan),['stress_level']),remainder='passthrough',verbose_feature_names_out=False)
trf1_test.set_output(transform="pandas")
x_test=trf1_test.fit_transform(x_test)

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

In [ ]:
scale_pos_weight=((y_train==0).sum()/(y_train==1).sum())

In [ ]:
xgbc = xgb.XGBClassifier(scale_pos_weight=scale_pos_weight,tree_method='hist',random_state=42)
param_grid = {
    "n_estimators": [1500,3000],
    "learning_rate": [0.005, 0.05, 0.01],
    "max_depth": [3,8],
    "subsample": [0.8,1.0],
}
grid_search_xgb=HalvingGridSearchCV(xgbc,cv=3,scoring='roc_auc',n_jobs=-1,factor=3,resource='n_samples',param_grid=param_grid,verbose=0)
grid_search_xgb.fit(x_train,y_train)

In [ ]:
print(grid_search_xgb.best_score_)
print(grid_search_xgb.best_params_)

In [ ]:
from lightgbm import LGBMClassifier
lgbm = LGBMClassifier(scale_pos_weight=scale_pos_weight,random_state=42, verbosity=-1)
param_grid = {
    "n_estimators": [1500,3000],
    "learning_rate": [0.005,0.05,0.01],
    "max_depth": [6,10],
    "subsample": [0.8, 1.0],
}
grid_search_lgbm=HalvingGridSearchCV(lgbm,cv=3,scoring='roc_auc',n_jobs=-1,factor=3,resource='n_samples',param_grid=param_grid,verbose=0)
grid_search_lgbm.fit(x_train,y_train)

In [ ]:
print(grid_search_lgbm.best_score_)
print(grid_search_lgbm.best_params_)

In [ ]:
from catboost import CatBoostClassifier
cb=CatBoostClassifier(auto_class_weights='Balanced')
params_cb= {
    "iterations": [1500,3000],
    "learning_rate": [0.005,0.05, 0.01],
    "depth": [6,8,10],
    "subsample"=: [0.8,1.0]
}
grid_search_cb=HalvingGridSearchCV(cb,cv=3,scoring='roc_auc',n_jobs=-1,factor=3,resource='n_samples',param_grid=params_cb,verbose=0)
grid_search_cb.fit(x_train,y_train)

In [ ]:
print(grid_search_cb.best_score_)
print(grid_search_cb.best_params_)

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score,train_test_split
from sklearn.metrics import roc_auc_score
estimators = [
    ("lgbm", LGBMClassifier(verbosity=-1,**grid_search_lgbm)),
    ("xgb", xgb.XGBClassifier(verbosity=-1,**grid_search_xgb)),
    ("cb",CatBoostClassifier(verbosity=-1,**grid_search_cb))
]

stacking_classifier = StackingClassifier(
    estimators=estimators, final_estimator=LogisticRegression(max_iter=500,class_weight='balanced'),n_jobs=-1,cv=3,passthrough=False,verbose=0
)
x_t,x_v,y_t,y_v=train_test_split(x_train,y_train,test_size=0.2,stratify=y_train, random_state=42)
stacking_classifier.fit(x_t,y_t)

In [ ]:
probs =stacking_classifier.predict_proba(x_v)[:, 1]
roc_auc_score(y_v, probs)

In [ ]:
df_submission=pd.DataFrame({'id':df_test['id'],'addicted_label':stacking_classifier.predict_proba(x_test.values)[:,1]})

In [ ]:
df_submission.index.name = None

In [ ]:
df_submission

In [ ]:
df_submission.to_csv('submission.csv',index=False)